## Climate Model and Climate Data Workshop - Hands-on Session
### Climate CH2025 example

This Jupyter notebook showcases how to download and use CH2025 data in Python. However, it is not a Python course!

https://www.meteoschweiz.admin.ch/klima/klimawandel/schweizer-klimaszenarien/berichte-daten-und-grafiken-der-klimaszenarien/klima-ch2025-datensaetze.html

https://opendatadocs.meteoswiss.ch/c-climate-data/c8-c9-climate-scenarios

In [ ]:
import xarray as xr
import glob

In [ ]:
variable = "pr" # choose pr or tas
ref = "ref91-20"
gwl = "gwl3.0" # choose a GWL, 1.5, 2.0, 2.5 or 3.0

In [ ]:
# the following path contains gridded data for the reference period 1991-2020, GWL1.5, GWL2.0, GWL2.5 and GWL3.0 for precipitation ("pr"), temperature ("tas")
# maximum daily temperature ("tasmax") and minimum daily temperature "tasmin" for all available models
path_ch2025 = f"/net/dust/c2sm-data/rlorenz/climate_model_data_workshop/ogd-climate-scenarios-ch2025-grid/{variable}"

If you want to use station instead of gridded data, the data would need to be downloaded first.
You can open a terminal and use the script `download_ch2025_interactive.sh` to download some station data. However, the rest of the exercises work with gridded data.

In [ ]:
# the filename pattern is as follows:
gwl_data = f"ogd-climate-scenarios-ch2025-grid_ch_{variable}_*_{gwl}.nc"

### Exercise 1

Calculate the difference between the reference period and GWL1.5 or GWL3.0 for each model

In [ ]:
diff_all_models_list = list()
# find all files that fit the filename pattern
for file in glob.glob(f"{path_ch2025}/{gwl_data}"):
    model_name = file.split("_")[6]
    print(model_name)
    with xr.open_dataset(file) as ds_gwl:
        # find matching ref data
        with xr.open_dataset(f"{path_ch2025}/ogd-climate-scenarios-ch2025-grid_ch_{variable}_{model_name}_{ref}.nc") as ds_ref:
            ds_diff = ds_gwl.mean(dim="time") - ds_ref.mean(dim="time")

    diff_all_models_list.append(ds_diff.assign_coords(model=model_name))

In [ ]:
# use xr.concat to concatenate the data from all models along the dimension "model"
ds_diff_all_models = xr.concat(diff_all_models_list, dim="model")

In [ ]:
ds_diff_all_models

### Exercise 2 

Calulate the multi-model median, the smallest and the largest difference.

In [ ]:
model_median = ds_diff_all_models.median(dim="model")

In [ ]:
model_min = ds_diff_all_models.min(dim="model")

In [ ]:
model_max = ds_diff_all_models.max(dim="model")

#### Exercise 2.1

Calculate the 5th and 95th quantiles (`quantile(0.05, dim="model")`).

In [ ]:
model_5th = ds_diff_all_models.quantile(0.05, dim="model")

In [ ]:
model_95th = ds_diff_all_models.quantile(0.95, dim="model")

### Exercise 3

Plot the median, smallest and largest difference

In [ ]:
# load packages
import cartopy.crs as ccrs
import cartopy.feature as cf
import matplotlib.pyplot as plt
import regionmask

In [ ]:
if variable == "pr":
    long_name = "precipitation"
elif variable == "tas":
    long_name = "temperature"
elif variable == "tasmax":
    long_name = "Maximum daily temperature"
elif variable == "tasmin":
    long_name = "Minimum daily temperature"
else:
    print(f"Error: Variable {variable} not implemented.")

In [ ]:
plt.figure(figsize=(10, 7))
proj=ccrs.LambertConformal(central_longitude=15)
# set up a map
ax = plt.axes(projection=proj)

model_median[variable].plot.pcolormesh(ax=ax, x="lon", y="lat", transform=ccrs.PlateCarree())
ax.add_feature(cf.BORDERS, edgecolor='lightgrey')
# fine tune the extent
ax.set_extent([5, 12, 45, 48.5], crs=ccrs.PlateCarree())
ax.set_title(f'Multi-model median {long_name} change {gwl} - {ref}')

In [ ]:
plt.figure(figsize=(10, 7))
proj=ccrs.LambertConformal(central_longitude=15)
# set up a map
ax = plt.axes(projection=proj)

model_min[variable].plot.pcolormesh(ax=ax, x="lon", y="lat", transform=ccrs.PlateCarree(), cmap = 'pink')
ax.add_feature(cf.BORDERS, edgecolor='lightgrey')
# fine tune the extent
ax.set_extent([5, 12, 45, 48.5], crs=ccrs.PlateCarree())
ax.set_title(f'Minimum {long_name} change {gwl} - {ref}')

In [ ]:
plt.figure(figsize=(10, 7))
proj=ccrs.LambertConformal(central_longitude=15)
# set up a map
ax = plt.axes(projection=proj)

model_max[variable].plot.pcolormesh(ax=ax, x="lon", y="lat", transform=ccrs.PlateCarree(), cmap = 'Blues')
ax.add_feature(cf.BORDERS, edgecolor='lightgrey')
# fine tune the extent
ax.set_extent([5, 12, 45, 48.5], crs=ccrs.PlateCarree())
ax.set_title(f'Maximum {long_name} change {gwl} - {ref}')

In [ ]:
plt.figure(figsize=(10, 7))
proj=ccrs.LambertConformal(central_longitude=15)
# set up a map
ax = plt.axes(projection=proj)

model_5th[variable].plot.pcolormesh(ax=ax, x="lon", y="lat", transform=ccrs.PlateCarree(), cmap = 'pink')
ax.add_feature(cf.BORDERS, edgecolor='lightgrey')
# fine tune the extent
ax.set_extent([5, 12, 45, 48.5], crs=ccrs.PlateCarree())
ax.set_title(f'5th quantile {long_name} change {gwl} - {ref}')

In [ ]:
plt.figure(figsize=(10, 7))
proj=ccrs.LambertConformal(central_longitude=15)
# set up a map
ax = plt.axes(projection=proj)

model_95th[variable].plot.pcolormesh(ax=ax, x="lon", y="lat", transform=ccrs.PlateCarree(), cmap = 'Blues')
ax.add_feature(cf.BORDERS, edgecolor='lightgrey')
# fine tune the extent
ax.set_extent([5, 12, 45, 48.5], crs=ccrs.PlateCarree())
ax.set_title(f'95th quantile {long_name} change {gwl} - {ref}')

#### Exercise 3.1

Plot the change between the reference period and the GWL for a specific model.

In [ ]:
model_name="mohc-hadrem-cnrm"

In [ ]:
plt.figure(figsize=(10, 7))
proj=ccrs.LambertConformal(central_longitude=15)
# set up a map
ax = plt.axes(projection=proj)

ds_diff_all_models.sel(model=model_name)[variable].plot.pcolormesh(ax=ax, x="lon", y="lat", transform=ccrs.PlateCarree(), cmap = 'RdBu')
ax.add_feature(cf.BORDERS, edgecolor='lightgrey')
# fine tune the extent
ax.set_extent([5, 12, 45, 48.5], crs=ccrs.PlateCarree())
ax.set_title(f'{long_name} change {gwl} - {ref} in model {model_name}')